# Lesson 17: Experiment hygiene (optional)

Goes with the Lesson 17 page of the Forkcast course. Seeds, a frozen config with a fingerprint, reproducible logged runs, grouping the log by configuration, and seed noise against a real difference.

Runtime: CPU is enough; every run takes well under a second.

## 1. Clone your repository and install the package

In [ ]:
GITHUB_USER = "your-github-username"   # edit this line
REPO = "forkcast-course"

!git clone https://github.com/{GITHUB_USER}/{REPO}.git
%cd {REPO}
!pip install -e .

## 2. `forkcast/experiments.py`

`git_commit` and `load_log` are provided. Write `seed_everything`, the `RunConfig` methods, `run_experiment`, `compare` and `repeat_with_seeds`.

In [ ]:
%%writefile forkcast/experiments.py
"""Lesson 17 (optional). Experiment hygiene: runs you can reproduce, compare and trust.

``git_commit`` and ``load_log`` are provided; the rest are exercises.
"""

from __future__ import annotations

import hashlib
import json
import random
import subprocess
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn


def seed_everything(seed: int, deterministic: bool = True) -> torch.Generator:
    """Seed every source of randomness a training run uses, and return a generator for DataLoaders.

    Python's ``random``, NumPy, torch (CPU and every CUDA device). With ``deterministic`` also call
    ``torch.use_deterministic_algorithms(deterministic, warn_only=True)`` and set
    ``torch.backends.cudnn.benchmark = not deterministic``. Return ``torch.Generator().manual_seed(seed)``,
    which a DataLoader takes as ``generator=`` so its shuffle order is reproducible too.
    """
    raise NotImplementedError("Lesson 17, step 2: random.seed, np.random.seed, torch.manual_seed, torch.cuda.manual_seed_all, determinism flags, return a Generator")


@dataclass(frozen=True)
class RunConfig:
    """Everything that decides a result, in one object. Change a field and the fingerprint changes."""
    model: str = "mlp"                      # "mlp" or "linear"
    hidden: tuple = (32, 32)
    lr: float = 0.01
    epochs: int = 150
    weight_decay: float = 0.0
    dropout: float = 0.0
    n_orders: int = 2000
    data_seed: int = 0
    seed: int = 0

    def to_json(self, path: str | Path) -> None:
        """Write the config as JSON (tuples become lists)."""
        raise NotImplementedError("Lesson 17, step 3: Path(path).write_text(json.dumps(asdict(self), indent=2))")

    @classmethod
    def from_json(cls, path: str | Path) -> "RunConfig":
        """Read a config back; turn the hidden list into a tuple."""
        raise NotImplementedError("Lesson 17, step 3: data = json.loads(...); data[\"hidden\"] = tuple(data[\"hidden\"]); return cls(**data)")

    def fingerprint(self) -> str:
        """First 8 hex characters of the SHA-256 of ``json.dumps(asdict(self), sort_keys=True)``: the run's identity."""
        raise NotImplementedError("Lesson 17, step 3: hashlib.sha256(json.dumps(asdict(self), sort_keys=True).encode()).hexdigest()[:8]")


def git_commit() -> str | None:
    """The current commit, so a result can be traced to the code that produced it; None outside a repository."""
    try:
        return subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, check=True, timeout=5).stdout.strip()
    except Exception:
        return None


def run_experiment(config: RunConfig, log_path: str | Path | None = None) -> dict:
    """Train one model from a config, reproducibly, and append the result to a JSON-lines log.

    seed_everything(config.seed); make_orders(config.n_orders, seed=config.data_seed, nonlinear=True);
    train_val_test_split(seed=config.data_seed); Normalizer fitted on the training part; the model is
    LinearRegressor(4) or MLP(4, hidden, dropout=...); fit_with_validation(epochs, lr, weight_decay).
    Returns {"fingerprint", "best_val", "best_epoch", "seconds", "commit", "timestamp", **asdict(config)}
    and, when log_path is given, appends it as one JSON line.
    """
    raise NotImplementedError("Lesson 17, step 4: seed, data, split, normalize, model, fit_with_validation, result dict, append a JSON line")


def load_log(path: str | Path) -> pd.DataFrame:
    """The experiment log as a DataFrame, one row per run."""
    return pd.read_json(path, lines=True)


def compare(log: pd.DataFrame) -> pd.DataFrame:
    """Group the log by everything but the seed and the per-run fields; report count, mean and std of best_val.

    Group keys: model, hidden (as a string), lr, epochs, weight_decay, dropout, n_orders, data_seed.
    Columns out: those keys plus runs, mean_val, std_val, sorted by mean_val.
    """
    raise NotImplementedError("Lesson 17, step 5: log.assign(hidden=...astype(str)).groupby(keys)[\"best_val\"].agg([\"count\", \"mean\", \"std\"])...")


def repeat_with_seeds(config: RunConfig, seeds, log_path: str | Path | None = None) -> dict:
    """Run the same config under several seeds. Returns {"values": [...], "mean": float, "std": float} (sample std, ddof=1)."""
    raise NotImplementedError("Lesson 17, step 5: dataclasses.replace(config, seed=s) for each seed; run_experiment; mean and std")

## 3. Seeds

In [ ]:
import importlib, random, torch
from dataclasses import replace
from torch.utils.data import DataLoader, TensorDataset
from forkcast import experiments as exp
from forkcast.model import MLP
importlib.reload(exp)

g1 = exp.seed_everything(123); a = MLP(4, (8,)); order1 = [b[0].tolist() for b in DataLoader(TensorDataset(torch.arange(16)), batch_size=4, shuffle=True, generator=g1)]
g2 = exp.seed_everything(123); b = MLP(4, (8,)); order2 = [b_[0].tolist() for b_ in DataLoader(TensorDataset(torch.arange(16)), batch_size=4, shuffle=True, generator=g2)]
print("same weights:", all(torch.equal(p, q) for p, q in zip(a.parameters(), b.parameters())), "| same shuffle:", order1 == order2)
print("deterministic algorithms:", torch.are_deterministic_algorithms_enabled(), "| cudnn.benchmark:", torch.backends.cudnn.benchmark)

## 4. Configs and fingerprints

In [ ]:
importlib.reload(exp)
cfg = exp.RunConfig(hidden=(16, 16), lr=0.005, seed=3)
cfg.to_json("run.json")
print("round trip equal:", exp.RunConfig.from_json("run.json") == cfg)
print("fingerprints:", cfg.fingerprint(), replace(cfg, lr=0.01).fingerprint(), replace(cfg, seed=4).fingerprint())

## 5. Reproducible, logged runs

In [ ]:
importlib.reload(exp)
import os
if os.path.exists("experiments.jsonl"):
    os.remove("experiments.jsonl")
a = exp.run_experiment(exp.RunConfig(), "experiments.jsonl")
b = exp.run_experiment(exp.RunConfig(), "experiments.jsonl")
print("same digits:", a["best_val"] == b["best_val"], "|", round(a["best_val"], 4), "in", round(a["seconds"], 2), "s | commit", a["commit"])
print(open("experiments.jsonl").readline()[:240])

## 6. Three experiments, five seeds each

In [ ]:
importlib.reload(exp)
import math
seeds = [0, 1, 2, 3, 4]
small = exp.repeat_with_seeds(exp.RunConfig(), seeds, "experiments.jsonl")
big = exp.repeat_with_seeds(replace(exp.RunConfig(), hidden=(64, 64)), seeds, "experiments.jsonl")
wd = exp.repeat_with_seeds(replace(exp.RunConfig(), weight_decay=1e-3), seeds, "experiments.jsonl")
for name, r in [("(32, 32)", small), ("(64, 64)", big), ("(32, 32) + weight decay", wd)]:
    print(f"{name:26s} {[round(v, 3) for v in r['values']]}  mean {r['mean']:.3f}  std {r['std']:.3f}")
print(exp.compare(exp.load_log("experiments.jsonl")).to_string())
print("spread of a 5-seed mean, small net:", round(small["std"] * math.sqrt(2 / 5), 3), "| (64,64) better by", round(small["mean"] - big["mean"], 3), "| weight decay better by", round(small["mean"] - wd["mean"], 3))

### Break it on purpose: a stray random call

In [ ]:
def leaky_run(config):
    idx = list(range(10)); random.shuffle(idx)          # unseeded randomness BEFORE seed_everything
    return exp.run_experiment(config)["best_val"], idx[:3]
r1 = leaky_run(exp.RunConfig()); r2 = leaky_run(exp.RunConfig())
print("the run itself still matches:", r1[0] == r2[0], "| but the pre-seed shuffle does not:", r1[1], r2[1])
print("anything downstream of that shuffle would drift between runs; move it after seed_everything and it is pinned")

## 7. Definition of done

Expected: `7 passed`, then `125 passed` for the whole course (the only skip is the GPU test).

In [ ]:
!pytest tests/test_17.py -q

In [ ]:
!pytest -m fast -q

## 8. Commit and push

In [ ]:
from getpass import getpass
TOKEN = getpass("GitHub token with access to your repo: ")

!git config user.name "{GITHUB_USER}"
!git config user.email "{GITHUB_USER}@users.noreply.github.com"
!git add -A
!git commit -m "Lesson 17: experiment hygiene"
!git push https://{TOKEN}@github.com/{GITHUB_USER}/{REPO}.git HEAD:main